In [ ]:
import mcfs
from mcfs.load_runs import load_run, load_runs

import numpy as np

import os
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = mcfs.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget
# %matplotlib inline

In [ ]:
gs, man, timing = load_run(
    base_load_path="/home/STORAGE/SKEWERS",
    simulation_name="TNG50-2",
    num=25,
    nspec=64,
    axis=-1,
    nbins=2048,
)

print("Loaded:", man["hdf5_path"])
print("z =", gs.red, "| NumLos =", gs.NumLos, "| nbins =", gs.nbins, "| dvbin =", gs.dvbin, "km/s")

if timing is not None:
    print("\n--- timing ---")
    print(timing)


In [ ]:
# --- choose lines ---
lines = [
    ("H",  1, 1215),
    ("Si", 3, 1206),
    ("Si", 2, 1190),
    ("Si", 2, 1193),
]
keys = [f"{e}{i}_{lam}" for (e, i, lam) in lines]

# --- choose which skewers to plot ---
# Options:
#   mode = "random"  -> random subset (seeded)
#   mode = "first"   -> first Nplot allowed skewers
#   mode = "given"   -> use ii_list exactly
mode = "random"

Nplot = 20
seed = 137

# Optional: restrict to LOS axis 1/2/3 (only relevant if gs.axis exists; for axis=-1 runs it does)
axis_select = None   # e.g. 1, 2, 3, or None

# For mode="given"
ii_list = [0, 1, 2, 3]   # only used if mode="given"

# --- set style ---
alpha = 0.6
lw = 1.0
ylog_tau = True

# --- allowed indices (optionally filtered by axis_select) ---
all_idx = np.arange(gs.NumLos)

if axis_select is not None:
    if not hasattr(gs, "axis"):
        raise AttributeError("gs has no attribute 'axis' (needed for axis_select).")
    allowed = np.where(gs.axis == axis_select)[0]
    if allowed.size == 0:
        raise ValueError(f"No skewers found with axis_select={axis_select}.")
else:
    allowed = all_idx

# --- choose ii_list ---
if mode == "random":
    rng = np.random.default_rng(seed)
    ii_list = rng.choice(allowed, size=min(Nplot, allowed.size), replace=False)
elif mode == "first":
    ii_list = allowed[: min(Nplot, allowed.size)]
elif mode == "given":
    ii_list = np.array(ii_list, dtype=int)
    bad = np.setdiff1d(ii_list, allowed)
    if bad.size > 0:
        raise ValueError(f"Some indices are not allowed (axis_select filter?): {bad[:10]}")
else:
    raise ValueError("mode must be one of: 'random', 'first', 'given'.")

print(f"Plotting {len(ii_list)} skewers | mode={mode} | axis_select={axis_select}")
print("Indices:", ii_list[:30], "..." if len(ii_list) > 30 else "")

# --- x-axis in velocity units (km/s) ---
x = np.arange(gs.nbins) * gs.dvbin
xlabel = r"Distance along LOS [$\mathrm{km\,s^{-1}}$]"

# --- load tau for each line ---
tau = {}
for elem, ion, lam in lines:
    key = f"{elem}{ion}_{lam}"
    tau[key] = gs.get_tau(elem, ion, lam)  # (NumLos, nbins)

# --- print mean flux across ALL skewers/pixels ---
print(f"\nGlobal means over all skewers and pixels (NumLos={gs.NumLos}, nbins={gs.nbins}):")
for k in keys:
    F = np.exp(-tau[k])
    print(f"  {k:10s}  <F> = {float(F.mean()):.6f}")

# --- plotting ---
cmap = plt.get_cmap("tab10")
colors = {k: cmap(i % cmap.N) for i, k in enumerate(keys)}

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Panel 1: tau
for k in keys:
    for ii in ii_list:
        axes[0].plot(x, tau[k][ii], color=colors[k], lw=lw, alpha=alpha)
axes[0].set_ylabel(r"$\tau$")
if ylog_tau:
    axes[0].set_yscale("log")
axes[0].set_title(f"{len(ii_list)} skewers | z={gs.red:.3f}")

# Panel 2: flux
for k in keys:
    for ii in ii_list:
        axes[1].plot(x, np.exp(-tau[k][ii]), color=colors[k], lw=lw, alpha=alpha)
axes[1].set_ylabel(r"$e^{-\tau}$")
axes[1].set_xlabel(xlabel)

# Legend: one entry per line
handles = [Line2D([0], [0], color=colors[k], lw=3, label=k) for k in keys]
axes[0].legend(handles=handles, loc="upper right", frameon=True, fontsize=12, title="Line key")

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------
# Pick a subset of N skewers
# -----------------------
N_plot_sample = 3
seed = 0
rng = np.random.default_rng(seed)
idx = np.arange(gs.NumLos) if N_plot_sample >= gs.NumLos else rng.choice(gs.NumLos, size=N_plot_sample, replace=False)
idx = np.array(idx, dtype=int)
print("Selected skewer indices:", idx)

# colors: one color per selected skewer, shared across both cases
cmap = plt.get_cmap("tab10")
colors = {ii: cmap(j % cmap.N) for j, ii in enumerate(idx)}

In [ ]:
x = np.arange(gs.nbins) * gs.dvbin
xlabel = r"Distance along LOS [$\mathrm{km\,s^{-1}}$]"

tau_lya = gs.get_tau("H", 1, 1215)  # (NumLos, nbins)

tau_tot = np.zeros((gs.NumLos, gs.nbins), dtype=np.float64)
for elem, ion, lam in lines:
    tau_tot += gs.get_tau(elem, ion, lam)

fig, (ax, axr) = plt.subplots(
    2, 1, figsize=(11, 6.5), sharex=True,
    gridspec_kw={"height_ratios": [3.2, 1.0], "hspace": 0.05}
)

# main panel: tau
for ii in idx:
    ax.plot(x, tau_tot[ii], color=colors[ii], lw=2.0, alpha=0.9, ls='--')
    ax.plot(x, tau_lya[ii], color=colors[ii], lw=1.0, alpha=0.9, ls='-')

ax.set_yscale("log")
ax.set_ylabel(r"$\tau$")
ax.set_title(f"z={gs.red:.3f}", fontsize=16)

# residual panel: log-residuals (avoid huge dynamic range)
for ii in idx:
    res = tau_tot[ii] - tau_lya[ii]
    axr.plot(x, res, color=colors[ii], lw=1.2, alpha=0.95, ls='-')

axr.set_yscale("log")
axr.set_xlabel(xlabel)
axr.set_ylabel(r"$\tau_\mathrm{all} - \tau_\mathrm{Ly\alpha}$")

# legend
h_case1 = Line2D([0],[0], color="k", lw=2.0, ls='-',  label=r"Ly$\alpha$ only")
h_case2 = Line2D([0],[0], color="k", lw=2.0, ls='--', label="All lines")
h_idx = [Line2D([0],[0], color=colors[ii], lw=3.0, label=f"skewer {ii}") for ii in idx]

ax.legend(handles=[h_case1, h_case2] + h_idx, loc="upper right", frameon=True, fontsize=10)

fig.tight_layout()
plt.show()

In [ ]:
x = np.arange(gs.nbins) * gs.dvbin
xlabel = r"Distance along LOS [$\mathrm{km\,s^{-1}}$]"

tau_lya = gs.get_tau("H", 1, 1215)  # (NumLos, nbins)

tau_tot = np.zeros((gs.NumLos, gs.nbins), dtype=np.float64)
for elem, ion, lam in lines:
    tau_tot += gs.get_tau(elem, ion, lam)

fig, (ax, axr) = plt.subplots(
    2, 1, figsize=(11, 6.5), sharex=True,
    gridspec_kw={"height_ratios": [3.2, 1.0], "hspace": 0.05}
)

# main panel
for ii in idx:
    ax.plot(x, np.exp(-tau_tot[ii]), color=colors[ii], lw=2.0, alpha=0.9, ls='--')
    ax.plot(x, np.exp(-tau_lya[ii]), color=colors[ii], lw=1.0, alpha=0.9, ls='-')

ax.set_ylabel(r"$e^{-\tau}$")
ax.set_title(f"z={gs.red:.3f}", fontsize=16)

# residual panel: log-residuals (avoid huge dynamic range)
for ii in idx:
    res = np.exp(-tau_lya[ii]) - np.exp(-tau_tot[ii])
    axr.plot(x, res, color=colors[ii], lw=1.2, alpha=0.95, ls='-')

axr.set_yscale("log")
axr.set_xlabel(xlabel)
axr.set_ylabel(r"$e^{-\tau_\mathrm{all}} - e^{-\tau_\mathrm{Ly\alpha}}$")

# legend
h_case1 = Line2D([0],[0], color="k", lw=2.0, ls='-',  label=r"Ly$\alpha$ only")
h_case2 = Line2D([0],[0], color="k", lw=2.0, ls='--', label="All lines")
h_idx = [Line2D([0],[0], color=colors[ii], lw=3.0, label=f"skewer {ii}") for ii in idx]
ax.legend(handles=[h_case1, h_case2] + h_idx, loc="upper right", frameon=True, fontsize=10)

fig.tight_layout()
plt.show()

In [ ]:
from mcfs.Xi1D import two_point_corr_1d, Xi1D_from_gs

In [ ]:
# -----------------------
# Compute Xi1D: Lyα only and "all lines"
# -----------------------
r_lya, xi_lya, _ = Xi1D_from_gs(gs, lines=[("H", 1, 1215)], n_bins=256)
r_all, xi_all, _ = Xi1D_from_gs(gs, lines=None, n_bins=256)  # uses manifest lines

In [ ]:
if not np.allclose(r_lya, r_all, atol=1e-12, rtol=0):
    raise ValueError("Different r grids for Lyα-only and all-lines. Ensure identical binning inputs.")
r = r_lya

m1, s1 = np.nanmean(xi_lya, axis=0), np.nanstd(xi_lya, axis=0)
m2, s2 = np.nanmean(xi_all, axis=0), np.nanstd(xi_all, axis=0)

In [ ]:
# -----------------------
# Plot controls
# -----------------------
xlim = (0.0, 3000.0)  # km/s; set None for full
mask = np.ones_like(r, dtype=bool)
if xlim is not None:
    mask &= (r >= xlim[0]) & (r <= xlim[1])

# -----------------------
# Figure 1: Xi1D comparison + residuals subpanel
#   - main: xi_all '--' and xi_lya '-'
#   - bottom: residual per skewer (xi_all - xi_lya) with shared x
# -----------------------
fig, (ax, axr) = plt.subplots(
    2, 1, figsize=(9, 7.0), sharex=True,
    gridspec_kw={"height_ratios": [3.2, 1.0], "hspace": 0.05}
)

# main panel: individual skewers
for ii in idx:
    ax.plot(r[mask], xi_all[ii, mask], color=colors[ii], lw=2.0, alpha=0.9, ls='--')
    ax.plot(r[mask], xi_lya[ii, mask], color=colors[ii], lw=1.0, alpha=0.9, ls='-')

# main panel: stacked mean ± std
ax.plot(r[mask], m2[mask], lw=2.0, c="k", ls='--')
ax.fill_between(r[mask], (m2 - s2)[mask], (m2 + s2)[mask], alpha=0.20, color="k")
ax.plot(r[mask], m1[mask], lw=1.0, c="k", ls='-')
ax.fill_between(r[mask], (m1 - s1)[mask], (m1 + s1)[mask], alpha=0.20, color="k")

ax.set_ylabel(r"$\xi(\Delta v)$")
ax.set_title(f"z={gs.red:.3f} | NumLos={gs.NumLos} | nbins={gs.nbins}", fontsize=16)

# residual panel: per skewer residuals (all - lya)
for ii in idx:
    res = xi_all[ii, mask] - xi_lya[ii, mask]
    axr.plot(r[mask], res, color=colors[ii], lw=1.2, alpha=0.95, ls='-')
axr.axhline(0.0, color="k", lw=1.0, alpha=0.6)
axr.set_xlabel(r"Separation $\Delta v$ [$\mathrm{km\,s^{-1}}$]")
axr.set_ylabel(r"$\Delta\xi$")
axr.grid(True, alpha=0.25)

ax.grid(True, alpha=0.25)

# legend: cases + sample index colors
h_case1 = Line2D([0],[0], color="k", lw=2.0, ls='-',  label=r"Ly$\alpha$ only")
h_case2 = Line2D([0],[0], color="k", lw=2.0, ls='--', label="All lines")
h_mean  = Line2D([0],[0], color="k", lw=3.0, label=r"Mean ± std")
h_idx   = [Line2D([0],[0], color=colors[ii], lw=3.0, label=f"skewer {ii}") for ii in idx]

ax.legend(handles=[h_case1, h_case2, h_mean] + h_idx, loc="upper right", frameon=True, fontsize=16)

fig.tight_layout()
plt.show()